In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")

print("Path to dataset files:", path)

100%|██████████| 4.07G/4.07G [00:45<00:00, 96.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/yelp-dataset/yelp-dataset/versions/4


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasserh/amazon-product-reviews-dataset")

print("Path to dataset files:", path)

100%|██████████| 708k/708k [00:00<00:00, 20.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/yasserh/amazon-product-reviews-dataset/versions/1


In [ ]:
import requests
import tarfile
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from scipy.sparse import csr_matrix, vstack
from scipy.linalg import sqrtm
from sklearn.decomposition import TruncatedSVD

url = "https://www.cs.jhu.edu/~mdredze/datasets/sentiment/processed_acl.tar.gz"
dataset_path = "processed_acl.tar.gz"
extracted_path = "processed_acl"

print(f"Downloading dataset from {url}...")
response = requests.get(url, stream=True)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
with open(dataset_path, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
print("Download complete.")

print(f"Extracting dataset to {extracted_path}...")
with tarfile.open(dataset_path, 'r:gz') as tar:
    # Add filter='data' to mitigate the DeprecationWarning and improve security
    tar.extractall(path=".", filter='data')
print("Extraction complete.")

Download complete.
Extracting dataset to processed_acl...
Extraction complete.


In [ ]:
def read_reviews(directory):
    all_reviews_content = []
    for sentiment in ['positive', 'negative', 'unlabeled']:
        file_path = os.path.join(directory, f'{sentiment}.review')
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
                all_reviews_content.extend([line.strip() for line in lines])
    return all_reviews_content

def read_labeled_reviews(directory):
    reviews = []
    labels = []
    for sentiment, label in [('positive', 1), ('negative', -1)]:
        file_path = os.path.join(directory, f'{sentiment}.review')
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
                reviews.extend(lines)
                labels.extend([label] * len(lines))
    return reviews, labels

source_domain = 'electronics'
target_domain = 'kitchen'

source_reviews, source_labels = read_labeled_reviews(os.path.join(extracted_path, source_domain))

all_target_reviews = read_reviews(os.path.join(extracted_path, target_domain))

labeled_target_reviews, labeled_target_labels = read_labeled_reviews(os.path.join(extracted_path, target_domain))

target_unlabeled_reviews_labeled_part, target_test_reviews, target_unlabeled_labels_part, target_test_labels = train_test_split(
    labeled_target_reviews, labeled_target_labels, test_size=400, stratify=labeled_target_labels, random_state=42)

target_unlabeled_reviews_from_file = read_reviews(os.path.join(extracted_path, target_domain, 'unlabeled.review'))
target_unlabeled_reviews = target_unlabeled_reviews_from_file + target_unlabeled_reviews_labeled_part


print(f"\nRead {len(source_reviews)} labeled reviews from the source domain ({source_domain}).")
print(f"Read {len(all_target_reviews)} all reviews from the target domain ({target_domain}).")
print(f"Created a labeled target test set of size {len(target_test_reviews)}.")
print(f"Using {len(target_unlabeled_reviews)} reviews as unlabeled data for the target domain.")


Read 2000 labeled reviews from the source domain (electronics).
Read 7945 all reviews from the target domain (kitchen).
Created a labeled target test set of size 400.
Using 1600 reviews as unlabeled data for the target domain.


In [ ]:
all_reviews = source_reviews + target_unlabeled_reviews

tfidf_vectorizer = TfidfVectorizer(max_features=5000)

tfidf_features = tfidf_vectorizer.fit_transform(all_reviews)

source_features = tfidf_features[:len(source_reviews)]
target_unlabeled_features = tfidf_features[len(source_reviews):]

print("\nTF-IDF feature extraction complete.")
print(f"Source features shape: {source_features.shape}")
print(f"Target unlabeled features shape: {target_unlabeled_features.shape}")

feature_names = tfidf_vectorizer.get_feature_names_out()


TF-IDF feature extraction complete.
Source features shape: (2000, 5000)
Target unlabeled features shape: (1600, 5000)


In [ ]:
source_features_dense = source_features.todense()
target_unlabeled_features_dense = target_unlabeled_features.todense()

source_word_counts = np.sum(source_features_dense > 0, axis=0)
target_word_counts = np.sum(target_unlabeled_features_dense > 0, axis=0)

source_word_counts = np.asarray(source_word_counts)[0]
target_word_counts = np.asarray(target_word_counts)[0]

min_freq = 10

domain_independent_indices = [i for i, name in enumerate(feature_names)
                              if source_word_counts[i] >= min_freq and target_word_counts[i] >= min_freq]

domain_independent_features = [feature_names[i] for i in domain_independent_indices]
domain_specific_features = [feature_names[i] for i in range(len(feature_names)) if i not in domain_independent_indices]

print(f"\nIdentified {len(domain_independent_features)} domain-independent features.")
print(f"Identified {len(domain_specific_features)} domain-specific features.")

domain_independent_indices_set = set(domain_independent_indices)
domain_specific_indices_all = [i for i in range(len(feature_names)) if i not in domain_independent_indices_set]


Identified 2077 domain-independent features.
Identified 2923 domain-specific features.


In [ ]:
import scipy
from scipy.sparse import lil_matrix, block_diag
from scipy.sparse.linalg import eigsh
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("\n--- Running Spectral Feature Alignment (SFA) ---")

# Initialize M as a sparse matrix for efficiency
M = scipy.sparse.lil_matrix((len(domain_specific_features), len(domain_independent_features)), dtype=float)

ds_feature_to_m_row_index = {name: i for i, name in enumerate(domain_specific_features)}
di_feature_to_m_col_index = {name: i for i, name in enumerate(domain_independent_features)}

# Iterate over the combined reviews to build the co-occurrence matrix
for review in all_reviews: # Corrected variable name
    word_counts = {}
    # Assuming review format is 'word:count word:count ...'
    for item in review.split():
        if ':' in item:
            word, count_str = item.split(':')
            try:
                word_counts[word] = int(count_str)
            except ValueError:
                continue

    present_ds_features = [word for word in word_counts if word in ds_feature_to_m_row_index]
    present_di_features = [word for word in word_counts if word in di_feature_to_m_col_index]

    for ds_word in present_ds_features:
        for di_word in present_di_features:
            row_idx = ds_feature_to_m_row_index[ds_word]
            col_idx = di_feature_to_m_col_index[di_word]
            M[row_idx, col_idx] += 1.0

M = M.tocsr() # Convert to CSR for efficient operations

print("Co-occurrence matrix M constructed.")
print(f"Shape of M: {M.shape}")

# Build the sparse block matrix A
top = scipy.sparse.hstack([scipy.sparse.csr_matrix((M.shape[0], M.shape[0])), M])
bottom = scipy.sparse.hstack([M.T, scipy.sparse.csr_matrix((M.shape[1], M.shape[1]))])
A = scipy.sparse.vstack([top, bottom]).tocsr()


# Compute the degree matrix D and its inverse square root
deg = np.array(A.sum(axis=1)).ravel()
deg_inv_sqrt = np.zeros_like(deg)
nonzero = deg > 0
deg_inv_sqrt[nonzero] = 1.0 / np.sqrt(deg[nonzero])
D_inv_sqrt = scipy.sparse.diags(deg_inv_sqrt)

# Compute the normalized affinity matrix L
L = D_inv_sqrt.dot(A).dot(D_inv_sqrt)   # normalized affinity
L = L.tocsr() # Ensure L is in CSR format

print("Matrix L constructed.")
print(f"Shape of L: {L.shape}")

K = 50

# Compute top-K eigenvectors using sparse eigenvalue decomposition
# 'LM' means largest magnitude eigenvalues
try:
    eigenvalues, eigenvectors = eigsh(L, k=K, which='LM')
    # eigsh returns complex numbers if L is not perfectly symmetric due to floating point errors; take real part
    eigenvectors = np.real(eigenvectors)
except Exception as e:
    print(f"Sparse eigsh failed: {e}. Falling back to dense computation (may be slow for large matrices).")
    L_dense = L.toarray()
    eigenvalues_dense, eigenvectors_dense = np.linalg.eigh(L_dense)
    sorted_indices_dense = np.argsort(eigenvalues_dense)[::-1] # Sort descending for largest
    eigenvectors = np.real(eigenvectors_dense[:, sorted_indices_dense[:K]])


U = eigenvectors  # shape (m_total, K) where m_total is len(ds) + len(di)

print(f"Selected {K} largest eigenvectors.")
print(f"Shape of U: {U.shape}")

# Split U into U_ds (for domain-specific features)
U_ds = U[:len(domain_specific_features), :]

print(f"Shape of U_ds (for domain-specific features): {U_ds.shape}")

# Select domain-specific features from the original TF-IDF matrices
source_ds_features = source_features[:, domain_specific_indices_all] # Keep as sparse
target_unlabeled_ds_features = target_unlabeled_features[:, domain_specific_indices_all] # Keep as sparse

# Project domain-specific features using U_ds
# Perform sparse matrix-dense matrix multiplication
source_ds_aligned = source_ds_features.dot(U_ds)
target_unlabeled_ds_aligned = target_unlabeled_ds_features.dot(U_ds)

print("Applied SFA mapping to domain-specific features.")
print(f"Shape of source_ds_aligned: {source_ds_aligned.shape}")
print(f"Shape of target_unlabeled_ds_aligned: {target_unlabeled_ds_aligned.shape}")

gamma = 0.5

# Augment original features with the aligned DS features
# Original features are the full TF-IDF vectors
source_features_augmented = scipy.sparse.hstack((source_features, gamma * source_ds_aligned)).tocsr()
target_unlabeled_features_augmented = scipy.sparse.hstack((target_unlabeled_features, gamma * target_unlabeled_ds_aligned)).tocsr()

print("Augmented features for SFA.")
print(f"Shape of source_features_augmented: {source_features_augmented.shape}")
print(f"Shape of target_unlabeled_features_augmented: {target_unlabeled_features_augmented.shape}")

# Train Logistic Regression classifier on augmented source features
sfa_classifier = LogisticRegression(C=10000, max_iter=1000) # Increased max_iter
sfa_classifier.fit(source_features_augmented, source_labels) # Use sparse matrix directly

print("\nLogistic Regression classifier trained on augmented source features.")

# Prepare target test data
target_test_features_tfidf = tfidf_vectorizer.transform(target_test_reviews)

# Select and project DS features for the test set
target_test_ds_features = target_test_features_tfidf[:, domain_specific_indices_all] # Keep as sparse
target_test_ds_aligned = target_test_ds_features.dot(U_ds)

# Augment target test features
target_test_features_augmented = scipy.sparse.hstack((target_test_features_tfidf, gamma * target_test_ds_aligned)).tocsr()

# Make predictions on the augmented target test features
target_test_predictions = sfa_classifier.predict(target_test_features_augmented) # Use sparse matrix directly

sfa_accuracy = accuracy_score(target_test_labels, target_test_predictions)

print("\nPredictions made on augmented target test features.")
print(f"Accuracy on target test set (SFA): {sfa_accuracy}")


--- Running Spectral Feature Alignment (SFA) ---
Co-occurrence matrix M constructed.
Shape of M: (2923, 2077)
Matrix L constructed.
Shape of L: (5000, 5000)
Selected 50 largest eigenvectors.
Shape of U: (5000, 50)
Shape of U_ds (for domain-specific features): (2923, 50)
Applied SFA mapping to domain-specific features.
Shape of source_ds_aligned: (2000, 50)
Shape of target_unlabeled_ds_aligned: (1600, 50)
Augmented features for SFA.
Shape of source_features_augmented: (2000, 5050)
Shape of target_unlabeled_features_augmented: (1600, 5050)

Logistic Regression classifier trained on augmented source features.

Predictions made on augmented target test features.
Accuracy on target test set (SFA): 0.975


In [ ]:
# --- BASELINE COMPARISON (Training/Testing on raw features without SFA) ---
# Baseline training uses the original raw features for the source domain
baseline_classifier = LogisticRegression(C=10000, max_iter=1000)
baseline_classifier.fit(source_features, source_labels)

# Prepare target test data using raw TF-IDF features
target_test_features_tfidf = tfidf_vectorizer.transform(target_test_reviews)

# Make predictions on the raw target test features
y_pred_baseline = baseline_classifier.predict(target_test_features_tfidf)
accuracy_baseline = accuracy_score(target_test_labels, y_pred_baseline)

print("\n--- BASELINE CLASSIFICATION RESULTS ---")
print(f"Baseline Accuracy (Raw Features Only): {accuracy_baseline:.4f}")

if sfa_accuracy > accuracy_baseline:
    print("\n✅ SFA improved the cross-domain classification accuracy!")
elif sfa_accuracy < accuracy_baseline:
     print("\n❌ SFA did not improve the accuracy in this test.")
else:
    print("\n😐 SFA achieved the same accuracy as the baseline in this test.")


--- BASELINE CLASSIFICATION RESULTS ---
Baseline Accuracy (Raw Features Only): 0.9750

😐 SFA achieved the same accuracy as the baseline in this test.


In [ ]:
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import vstack
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

print("\n--- Running LSA and FALSA Baselines ---")

n_components_lsa = K
svd = TruncatedSVD(n_components=n_components_lsa, random_state=42)

lsa_features = svd.fit_transform(tfidf_features)

source_features_lsa = lsa_features[:len(source_reviews)]
target_unlabeled_features_lsa = lsa_features[len(source_reviews):]

print(f"LSA transformation complete with {n_components_lsa} components.")
print(f"Source features shape (LSA): {source_features_lsa.shape}")
print(f"Target unlabeled features shape (LSA): {target_unlabeled_features_lsa.shape}")

lsa_classifier = LogisticRegression(C=10000)
lsa_classifier.fit(source_features_lsa, source_labels)

print("Logistic Regression classifier (LSA baseline) trained.")

target_test_features_tfidf_lsa = tfidf_vectorizer.transform(target_test_reviews)
target_test_features_lsa = svd.transform(target_test_features_tfidf_lsa)

lsa_target_test_predictions = lsa_classifier.predict(target_test_features_lsa)

lsa_accuracy = accuracy_score(target_test_labels, lsa_target_test_predictions)

print("\nPredictions made on LSA-transformed target test features.")
print(f"Accuracy on target test set (LSA baseline): {lsa_accuracy}")

source_ds_features_tfidf = source_features[:, domain_specific_indices_all]
target_unlabeled_ds_features_tfidf = target_unlabeled_features[:, domain_specific_indices_all]
target_test_ds_features_tfidf = target_test_features_tfidf_lsa[:, domain_specific_indices_all]

all_ds_features_tfidf = vstack([source_ds_features_tfidf, target_unlabeled_ds_features_tfidf])
svd_falsa = TruncatedSVD(n_components=n_components_lsa, random_state=42)
falsa_ds_features = svd_falsa.fit_transform(all_ds_features_tfidf)

source_ds_features_falsa = falsa_ds_features[:source_ds_features_tfidf.shape[0]]
target_unlabeled_ds_features_falsa = falsa_ds_features[source_ds_features_tfidf.shape[0]:]

print(f"\nFALSA transformation complete for domain-specific features with {n_components_lsa} components.")
print(f"Source DS features shape (FALSA): {source_ds_features_falsa.shape}")
print(f"Target unlabeled DS features shape (FALSA): {target_unlabeled_ds_features_falsa.shape}")

source_features_falsa_augmented = np.hstack((source_features.todense(), gamma * source_ds_features_falsa))
target_test_features_falsa_augmented = np.hstack((target_test_features_tfidf_lsa.todense(), gamma * svd_falsa.transform(target_test_ds_features_tfidf)))

print("Augmented features for FALSA.")
print(f"Shape of source_features_falsa_augmented: {source_features_falsa_augmented.shape}")
print(f"Shape of target_test_features_falsa_augmented: {target_test_features_falsa_augmented.shape}")

falsa_classifier = LogisticRegression(C=10000)
falsa_classifier.fit(np.asarray(source_features_falsa_augmented), source_labels)

print("Logistic Regression classifier (FALSA baseline) trained.")

falsa_target_test_predictions = falsa_classifier.predict(np.asarray(target_test_features_falsa_augmented))

falsa_accuracy = accuracy_score(target_test_labels, falsa_target_test_predictions)

print("\nPredictions made on augmented target test features using FALSA baseline.")
print(f"Accuracy on target test set (FALSA baseline): {falsa_accuracy}")


--- Running LSA and FALSA Baselines ---
LSA transformation complete with 50 components.
Source features shape (LSA): (2000, 50)
Target unlabeled features shape (LSA): (1600, 50)
Logistic Regression classifier (LSA baseline) trained.

Predictions made on LSA-transformed target test features.
Accuracy on target test set (LSA baseline): 0.9475

FALSA transformation complete for domain-specific features with 50 components.
Source DS features shape (FALSA): (2000, 50)
Target unlabeled DS features shape (FALSA): (1600, 50)
Augmented features for FALSA.
Shape of source_features_falsa_augmented: (2000, 5050)
Shape of target_test_features_falsa_augmented: (400, 5050)
Logistic Regression classifier (FALSA baseline) trained.

Predictions made on augmented target test features using FALSA baseline.
Accuracy on target test set (FALSA baseline): 0.975


In [ ]:
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack, csr_matrix
import numpy as np

print("\n--- Running SCL Baseline ---")

source_labels_np = np.array(source_labels)

correlations = []
for i in range(source_features_dense.shape[1]):
    feature_vector = np.asarray(source_features_dense[:, i]).flatten()
    if np.std(feature_vector) > 1e-6:
        correlation = np.corrcoef(feature_vector, source_labels_np)[0, 1]
        correlations.append(abs(correlation))
    else:
        correlations.append(0)

correlations = np.array(correlations)

min_freq_pivot = 10
min_correlation = 0.05

pivot_indices = [i for i, name in enumerate(feature_names)
                 if source_word_counts[i] >= min_freq_pivot and
                    target_word_counts[i] >= min_freq_pivot and
                    correlations[i] >= min_correlation]

pivot_features = [feature_names[i] for i in pivot_indices]
non_pivot_indices = [i for i in range(len(feature_names)) if i not in pivot_indices]
non_pivot_features = [feature_names[i] for i in non_pivot_indices]

print(f"Identified {len(pivot_features)} pivot features.")
print(f"Identified {len(non_pivot_features)} non-pivot features.")

source_pivot_features = source_features[:, pivot_indices]
source_non_pivot_features = source_features[:, non_pivot_indices]
target_unlabeled_pivot_features = target_unlabeled_features[:, pivot_indices]
target_unlabeled_non_pivot_features = target_unlabeled_features[:, non_pivot_indices]

print(f"Source pivot features shape: {source_pivot_features.shape}")
print(f"Source non-pivot features shape: {source_non_pivot_features.shape}")
print(f"Target unlabeled pivot features shape: {target_unlabeled_pivot_features.shape}")
print(f"Target unlabeled non-pivot features shape: {target_unlabeled_non_pivot_features.shape}")

all_pivot_features = vstack([source_pivot_features, target_unlabeled_pivot_features])

P_columns = []

print("\nTraining linear classifiers for non-pivot features...")
all_reviews_features = vstack([source_features, target_unlabeled_features])

for i, non_pivot_idx in enumerate(non_pivot_indices):
    non_pivot_target = np.asarray(all_reviews_features[:, non_pivot_idx].todense()).flatten() > 0

    correspondence_classifier = LogisticRegression(C=1.0, solver='liblinear')
    correspondence_classifier.fit(all_pivot_features, non_pivot_target)

    P_columns.append(correspondence_classifier.coef_[0])

    if (i + 1) % 100 == 0:
        print(f"Trained classifier for {i + 1}/{len(non_pivot_indices)} non-pivot features.")

print("Finished training classifiers for non-pivot features.")

P = np.array(P_columns).T

print(f"\nTransformation matrix P constructed with shape: {P.shape}")

target_test_features_tfidf_scl = tfidf_vectorizer.transform(target_test_reviews)

target_test_pivot_features = target_test_features_tfidf_scl[:, pivot_indices]
target_test_non_pivot_features = target_test_features_tfidf_scl[:, non_pivot_indices]

source_transformed_non_pivot = source_pivot_features.dot(P)

target_unlabeled_transformed_non_pivot = target_unlabeled_pivot_features.dot(P)

target_test_transformed_non_pivot = target_test_pivot_features.dot(P)

print("Applied SCL transformation to non-pivot features.")
print(f"Shape of source_transformed_non_pivot: {source_transformed_non_pivot.shape}")
print(f"Shape of target_unlabeled_transformed_non_pivot: {target_unlabeled_transformed_non_pivot.shape}")
print(f"Shape of target_test_transformed_non_pivot: {target_test_transformed_non_pivot.shape}")

source_features_dense_original_scl = source_features.todense()
target_test_features_dense_original_scl = target_test_features_tfidf_scl.todense()

source_features_scl_augmented = np.hstack((source_features_dense_original_scl, source_transformed_non_pivot))
target_test_features_scl_augmented = np.hstack((target_test_features_dense_original_scl, target_test_transformed_non_pivot))

print("Augmented features for SCL.")
print(f"Shape of source_features_scl_augmented: {source_features_scl_augmented.shape}")
print(f"Shape of target_test_features_scl_augmented: {target_test_features_scl_augmented.shape}")

scl_classifier = LogisticRegression(C=10000)
scl_classifier.fit(np.asarray(source_features_scl_augmented), source_labels)

print("\nLogistic Regression classifier (SCL baseline) trained on augmented source features.")

scl_target_test_predictions = scl_classifier.predict(np.asarray(target_test_features_scl_augmented))

scl_accuracy = accuracy_score(target_test_labels, scl_target_test_predictions)

print("\nPredictions made on augmented target test features using SCL baseline.")
print(f"Accuracy on target test set (SCL baseline): {scl_accuracy}")


--- Running SCL Baseline ---
Identified 537 pivot features.
Identified 4463 non-pivot features.
Source pivot features shape: (2000, 537)
Source non-pivot features shape: (2000, 4463)
Target unlabeled pivot features shape: (1600, 537)
Target unlabeled non-pivot features shape: (1600, 4463)

Training linear classifiers for non-pivot features...
Trained classifier for 100/4463 non-pivot features.
Trained classifier for 200/4463 non-pivot features.
Trained classifier for 300/4463 non-pivot features.
Trained classifier for 400/4463 non-pivot features.
Trained classifier for 500/4463 non-pivot features.
Trained classifier for 600/4463 non-pivot features.
Trained classifier for 700/4463 non-pivot features.
Trained classifier for 800/4463 non-pivot features.
Trained classifier for 900/4463 non-pivot features.
Trained classifier for 1000/4463 non-pivot features.
Trained classifier for 1100/4463 non-pivot features.
Trained classifier for 1200/4463 non-pivot features.
Trained classifier for 1300

In [ ]:
print("\n--- Running NoTransf Baseline ---")

no_transf_classifier = LogisticRegression(C=10000)
no_transf_classifier.fit(np.asarray(source_features.todense()), source_labels)

print("Logistic Regression classifier (NoTransf baseline) trained on original source features.")

target_test_features_original = tfidf_vectorizer.transform(target_test_reviews)

no_transf_target_test_predictions = no_transf_classifier.predict(np.asarray(target_test_features_original.todense()))

no_transf_accuracy = accuracy_score(target_test_labels, no_transf_target_test_predictions)

print("Predictions made on original target test features using NoTransf baseline.")
print(f"Accuracy on target test set (NoTransf baseline): {no_transf_accuracy}")


--- Running NoTransf Baseline ---
Logistic Regression classifier (NoTransf baseline) trained on original source features.
Predictions made on original target test features using NoTransf baseline.
Accuracy on target test set (NoTransf baseline): 0.975
